In [20]:
import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from itertools import combinations
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print("Imports done")

Imports done


In [21]:
# ── Load labeled puzzles ─────────────────────────────────────────────
with open("../data/puzzles_labeled.json", "r") as f:
    puzzles = json.load(f)

print(f"Loaded {len(puzzles)} labeled puzzles")

# ── Fix words list to come from groups (ensures consistency) ─────────
for puzzle in puzzles:
    words_from_groups = []
    for group in puzzle["groups"]:
        words_from_groups.extend(group["members"])
    puzzle["words"] = words_from_groups

# ── Remove April Fools emoji puzzle ──────────────────────────────────
puzzles = [p for p in puzzles if p["puzzle_id"] != 295]
print(f"Puzzles after removing emoji puzzle: {len(puzzles)}")

# ── Load MPNet and compute embeddings ────────────────────────────────
print("Loading MPNet model...")
model = SentenceTransformer("all-mpnet-base-v2")
print("Model loaded")

print("Computing embeddings...")
for puzzle in puzzles:
    embeddings = model.encode(puzzle["words"], show_progress_bar=False)
    puzzle["embeddings"] = embeddings

print(f"Embeddings computed for {len(puzzles)} puzzles")

Loaded 227 labeled puzzles
Puzzles after removing emoji puzzle: 226
Loading MPNet model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded
Computing embeddings...
Embeddings computed for 226 puzzles


In [22]:
# ── Color mapping ────────────────────────────────────────────────────
COLORS = {0: "yellow", 1: "green", 2: "blue", 3: "purple"}
COLOR_HEX = {0: "#F9DF6D", 1: "#A0C35A", 2: "#B0C4EF", 3: "#BA81C5"}

def get_group_embeddings(puzzle):
    """Returns dict of {level: {words, embeddings}} for each group."""
    word_list = puzzle["words"]
    emb_array = np.array(puzzle["embeddings"])
    word_to_emb = {w: emb_array[i] for i, w in enumerate(word_list)}

    groups = {}
    for group in puzzle["groups"]:
        level = group["level"]
        members = group["members"]
        embs = np.array([word_to_emb[w] for w in members])
        groups[level] = {"words": members, "embeddings": embs}
    return groups

def cohesion(embeddings):
    """Mean pairwise cosine similarity within a group."""
    sims = cosine_similarity(embeddings)
    n = len(embeddings)
    total = sum(sims[i][j] for i in range(n) for j in range(i+1, n))
    return total / (n * (n - 1) / 2)

def confusability(group_embs, other_embs):
    """Mean cosine similarity between a group and all other words."""
    sims = cosine_similarity(group_embs, other_embs)
    return sims.mean()

def puzzle_metrics(puzzle):
    """Compute cohesion, confusability, and silhouette for a puzzle."""
    groups = get_group_embeddings(puzzle)
    all_embs = np.array(puzzle["embeddings"])
    all_words = puzzle["words"]

    word_to_level = {}
    for group in puzzle["groups"]:
        for w in group["members"]:
            word_to_level[w] = group["level"]
    labels = [word_to_level[w] for w in all_words]

    results = {}
    for level, data in groups.items():
        g_embs = data["embeddings"]
        other_embs = np.vstack([
            groups[l]["embeddings"] for l in groups if l != level
        ])
        results[level] = {
            "cohesion": cohesion(g_embs),
            "confusability": confusability(g_embs, other_embs)
        }

    sil = silhouette_score(all_embs, labels, metric="cosine")
    mean_coh = np.mean([r["cohesion"] for r in results.values()])
    mean_conf = np.mean([r["confusability"] for r in results.values()])

    return {
        "mean_cohesion": mean_coh,
        "mean_confusability": mean_conf,
        "silhouette": sil,
        "by_group": results
    }

print("Metric functions defined")

Metric functions defined


In [23]:
# ── Compute metrics for all puzzles ─────────────────────────────────
records = []
for puzzle in puzzles:
    metrics = puzzle_metrics(puzzle)
    records.append({
        "puzzle_id": puzzle["puzzle_id"],
        "date": puzzle["date"],
        "difficulty": puzzle["difficulty"],
        "mean_cohesion": metrics["mean_cohesion"],
        "mean_confusability": metrics["mean_confusability"],
        "silhouette": metrics["silhouette"],
        "by_group": metrics["by_group"]
    })

df = pd.DataFrame(records)

# ── Correlations table (Table 4.1) ───────────────────────────────────
corr_cols = ["difficulty", "mean_cohesion", "mean_confusability", "silhouette"]
print("Pairwise correlations:")
print(df[corr_cols].corr().round(3))

Pairwise correlations:
                    difficulty  mean_cohesion  mean_confusability  silhouette
difficulty               1.000         -0.196               0.054      -0.240
mean_cohesion           -0.196          1.000               0.346       0.823
mean_confusability       0.054          0.346               1.000      -0.206
silhouette              -0.240          0.823              -0.206       1.000
